# Experimentation: A/B Testing Done Properly

Notebook for **Chapter 7** of the *ds* study document. Run the cells top to bottom. Read the full explanation on the site: https://dadipavan.github.io/Learning/study/ds/ch07.html

Tip: change a number, predict the output, then run the cell again.

## 7.1 Designing the Test and Choosing the Sample Size

In [ ]:
import math
from scipy import stats

def n_per_arm(p1, p2, alpha=0.05, power=0.8):
    za = stats.norm.ppf(1 - alpha/2); zb = stats.norm.ppf(power)
    pbar = (p1 + p2) / 2
    num = (za*math.sqrt(2*pbar*(1-pbar)) + zb*math.sqrt(p1*(1-p1) + p2*(1-p2)))**2
    return num / (p2 - p1)**2

print(n_per_arm(.10, .11))    # 14750.79
print(n_per_arm(.10, .12))    # 3840.85
print(n_per_arm(.10, .105))   # 57762.65

## 7.2 Analysing the Result and Cutting Noise with CUPED

In [ ]:
import numpy as np, math
rng = np.random.default_rng(42); N = 20000
pre = rng.gamma(2, 150, N)
noise = rng.normal(0, 200, N)
y0 = 0.8*pre + noise + 100
T = rng.integers(0, 2, N)
y = y0 + 15*T

def diff_se(y, T):
    a, b = y[T==1], y[T==0]
    return a.mean()-b.mean(), math.sqrt(a.var(ddof=1)/len(a) + b.var(ddof=1)/len(b))

print(diff_se(y, T))
theta = np.cov(y, pre)[0,1] / pre.var(ddof=1)
yc = y - theta*(pre - pre.mean())
print(theta, np.corrcoef(y, pre)[0,1])
print(diff_se(yc, T))

## 7.3 Pitfalls: SRM, Peeking and Awkward Metrics

In [ ]:
import numpy as np
rng = np.random.default_rng(2); sims = 20000; n = 10000
c = np.arange(1, 11) * 1000
peek = end = poc = 0
for _ in range(sims):
    a = np.cumsum(rng.binomial(1, .1, n))[c-1]
    b = np.cumsum(rng.binomial(1, .1, n))[c-1]
    p = (a+b)/(2*c); se = np.sqrt(2*p*(1-p)/c)
    z = np.abs((b-a)/c/se)
    peek += (z > 1.96).any(); end += z[-1] > 1.96; poc += (z > 2.555).any()
print(peek/sims, end/sims, poc/sims)

## 7.4 Sequential Tests, Bandits and the Decision

In [ ]:
import numpy as np
true = np.array([.10, .12, .11]); T_ = 10000
def run(policy, rng):
    s = np.ones(3); f = np.ones(3); rew = 0; pulls = np.zeros(3)
    for t in range(T_):
        arm = int(np.argmax(rng.beta(s, f))) if policy == "ts" else t % 3
        r = rng.random() < true[arm]; rew += r
        s[arm] += r; f[arm] += 1 - r; pulls[arm] += 1
    return rew, pulls

## 7.5 Ratio Metrics and the Delta Method

Many headline metrics are ratios: click-through rate is clicks per session, revenue per session, orders per visit. The unit you randomise is the *user*, but the denominator counts *sessions*. Users with many sessions contribute many rows, and a user's sessions are not independent (a person who likes clicking clicks in most sessions). If you treat each session as an independent coin flip you get a standard error that is too small and too many false wins.

In [ ]:
import numpy as np
# Tiny hand example: 5 users in one arm
s = np.array([1, 2, 4, 3, 10.0])      # sessions
c = np.array([0, 1, 1, 1, 4.0])       # clicks
n = len(s)
R = c.sum() / s.sum()
ms, mc = s.mean(), c.mean()
vs, vc = s.var(ddof=1), c.var(ddof=1)
cov = np.cov(c, s, ddof=1)[0, 1]
var_R = (vc - 2 * R * cov + R**2 * vs) / (n * ms**2)
print("R", R, "var_c", vc, "var_s", vs, "cov", cov)
print("delta var", round(var_R, 5), "se", round(var_R**.5, 4))
rng = np.random.default_rng(0)
idx = rng.integers(0, n, (20000, n))
boot = c[idx].sum(1) / s[idx].sum(1)
print("bootstrap se", boot.std().round(4))

R 0.35 var_c 2.3 var_s 12.5 cov 5.25
delta var 0.00195 se 0.0442
bootstrap se 0.0485


In [ ]:
import numpy as np
rng = np.random.default_rng(1)
n = 20000
# heavy users have many sessions AND a user-specific click rate (users are clustered)
rate = rng.beta(2, 18, n)                        # user click propensity, mean 0.10
sess = rng.poisson(1, n) + 1 + (rng.random(n) < 0.05) * rng.poisson(30, n)
clicks = rng.binomial(sess, rate)
R = clicks.sum() / sess.sum()
# naive: treat every session as independent Bernoulli
se_naive = np.sqrt(R * (1 - R) / sess.sum())
# delta method at user level
ms = sess.mean()
cov = np.cov(clicks, sess)[0, 1]
var_R = (clicks.var(ddof=1) - 2 * R * cov + R**2 * sess.var(ddof=1)) / (n * ms**2)
# bootstrap users
idx = rng.integers(0, n, (400, n))
boot = clicks[idx].sum(1) / sess[idx].sum(1)
print("CTR", round(R, 4), "sessions per user", round(ms, 2))
print("naive SE", round(se_naive, 5), " delta SE", round(var_R**.5, 5), " bootstrap SE", round(boot.std(), 5))
print("naive understates SE by factor", round(var_R**.5 / se_naive, 2))

CTR 0.0963 sessions per user 3.48
naive SE 0.00112  delta SE 0.00146  bootstrap SE 0.00152
naive understates SE by factor 1.3


## 7.6 Sample Size for Non-Binary Metrics and Smarter Variance Reduction

Section 7.1 sized a yes/no metric. The rule of thumb `n ≈ 16 p(1−p)/δ²` assumes a two-sided 5% test and 80% power (`2·(1.96 + 0.84)² = 15.7`, rounded to 16). For other settings change the 15.7: a one-sided 5% test needs about 12.4 (`2·(1.645+0.842)²`), and 90% power at two-sided 5% needs about 21.0 (`2·(1.96+1.282)²`). In general, for a metric with standard deviation `σ` per user, `n per arm = 2·(zα/2 + zβ)²·σ² / δ²`.

In [ ]:
from scipy import stats
za, zb = stats.norm.ppf(0.975), stats.norm.ppf(0.8)
k = 2 * (za + zb) ** 2
print("2*(za+zb)^2 =", round(k, 3))
# continuous: revenue per user mean 50, sd 120, detect +2
sd, delta = 120, 2
print("continuous n per arm", round(k * sd**2 / delta**2))
# CUPED with rho = 0.5 cuts variance by 25%
print("with rho 0.5:", round(k * sd**2 * (1 - 0.25) / delta**2))
# ratio metric: delta-method variance per user of the linearised metric
# CTR 0.10, sd of linearised value per user = 0.35, detect +0.005
print("ratio n per arm", round(k * 0.35**2 / 0.005**2))
# one-sided 5% and power 90% for the binary 10% vs 11% case
def n_bin(p1, p2, alpha, power, two=True):
    za = stats.norm.ppf(1 - alpha / (2 if two else 1)); zb = stats.norm.ppf(power)
    pb = (p1 + p2) / 2
    return (za*(2*pb*(1-pb))**.5 + zb*(p1*(1-p1)+p2*(1-p2))**.5)**2 / (p2-p1)**2
print("binary 10 to 11:", round(n_bin(.10,.11,.05,.8)), round(n_bin(.10,.11,.05,.8,False)), round(n_bin(.10,.11,.05,.9)))

2*(za+zb)^2 = 15.698
continuous n per arm 56512
with rho 0.5: 42384
ratio n per arm 76919
binary 10 to 11: 14751 11619 19747


In [ ]:
import numpy as np
rng = np.random.default_rng(3)
n = 8000
pre = rng.gamma(2, 25, n)                        # last quarter spend
seg = (rng.random(n) < 0.3).astype(int)          # 1 = premium segment
y0 = 0.6 * pre + 40 * seg + rng.normal(0, 30, n)
def one_trial(rng):
    t = rng.integers(0, 2, n)
    y = y0 + 3 * t + rng.normal(0, 5, n)         # true effect +3
    plain = y[t == 1].mean() - y[t == 0].mean()
    # post-stratified by segment
    est = 0
    for g in (0, 1):
        m = seg == g
        est += m.mean() * (y[m & (t == 1)].mean() - y[m & (t == 0)].mean())
    # regression adjustment: y ~ t + pre + seg
    X = np.c_[np.ones(n), t, pre - pre.mean(), seg - seg.mean()]
    beta = np.linalg.lstsq(X, y, rcond=None)[0]
    return plain, est, beta[1]
res = np.array([one_trial(rng) for _ in range(600)])
print("mean estimate (truth 3):", res.mean(0).round(2))
print("empirical SE: plain, stratified, regression:", res.std(0).round(2))
print("variance vs plain:", (res.var(0) / res.var(0)[0]).round(2))

mean estimate (truth 3): [3.1  3.06 3.04]
empirical SE: plain, stratified, regression: [0.9  0.79 0.65]
variance vs plain: [1.   0.77 0.52]


## 7.7 Many Metrics: Guardrails and False Discovery Control

A real test tracks 20 or 30 metrics. At a 5% level each, a null test shows about one "significant" metric by luck. Two standard fixes: **Bonferroni** (use `α/m` for each of `m` metrics; strict, controls the chance of *any* false alarm) and **Benjamini-Hochberg (BH)**, which controls the expected share of false discoveries among the metrics you call significant (the false discovery rate, FDR).

In [ ]:
import numpy as np
p = np.array([0.001, 0.008, 0.012, 0.041, 0.049, 0.2, 0.3, 0.5, 0.7, 0.9])
m, q = len(p), 0.05
order = np.argsort(p); ps = p[order]
thr = q * np.arange(1, m + 1) / m
ok = ps <= thr
k = np.max(np.where(ok)[0]) + 1 if ok.any() else 0
print("BH thresholds", thr.round(3)[:5], "largest k", k, "rejected", ps[:k])
print("Bonferroni rejects", (p <= q / m).sum())
adj = np.minimum.accumulate((ps * m / np.arange(1, m + 1))[::-1])[::-1]
print("BH adjusted p-values", adj.round(3))

BH thresholds [0.005 0.01  0.015 0.02  0.025] largest k 3 rejected [0.001 0.008 0.012]
Bonferroni rejects 1
BH adjusted p-values [0.01  0.04  0.04  0.098 0.098 0.333 0.429 0.625 0.778 0.9  ]


## 7.8 When Users Affect Each Other: Interference, Switchbacks and Geo Tests

A/B testing assumes each user's outcome depends only on their own assignment (no interference, the "stable unit" assumption). That fails in marketplaces (drivers, riders, inventory), social products (friends see each other's changes) and auctions (ad budgets). Then user-level randomisation can show a lift that does not exist after launch.

In [ ]:
import numpy as np
rng = np.random.default_rng(0)
N, C, p0, p1 = 100000, 45000, 0.5, 0.6
def served(requests): return min(1.0, C / requests)       # share of requests that get a slot
# truth by rolling out to everyone
ctrl = p0 * served(N * p0) ; trt = p1 * served(N * p1)
print("global control orders per user", round(ctrl, 3), " global treatment", round(trt, 3), " true effect", round(trt - ctrl, 3) + 0.0)
# user-level A/B inside the same market
treated = rng.random(N) < 0.5
req = rng.random(N) < np.where(treated, p1, p0)
sv = served(req.sum())
order = req & (rng.random(N) < sv)
print("A/B measured: control", round(order[~treated].mean(), 3), "treatment", round(order[treated].mean(), 3),
      "lift", round(order[treated].mean() - order[~treated].mean(), 3))
# switchback: whole market alternates by day, 28 days
days = np.arange(28); tday = days % 2 == 1
o = []
for t in tday:
    r = rng.random(N) < (p1 if t else p0)
    o.append((r & (rng.random(N) < served(r.sum()))).mean())
o = np.array(o)
print("switchback estimate", round(o[tday].mean() - o[~tday].mean(), 3) + 0.0)

global control orders per user 0.45  global treatment 0.45  true effect 0.0
A/B measured: control 0.409 treatment 0.492 lift 0.083
switchback estimate 0.0


## 7.9 Looking Early Without Cheating: Group-Sequential and Always-Valid Tests

Section 7.3 showed that peeking at a fixed-sample test inflates false positives. The answer is not "never look". It is to use a method whose error rate is correct *because it expects you to look*. Two families: group-sequential designs (a few planned looks with stricter cut-offs) and always-valid tests such as the mixture sequential probability ratio test (mSPRT), which allows continuous monitoring.

In [ ]:
import numpy as np
rng = np.random.default_rng(5)
paths = 400000
z = np.cumsum(rng.normal(size=(paths, 5)), axis=1) / np.sqrt(np.arange(1, 6))   # z-stat at each of 5 equal looks
naive = (np.abs(z) > 1.96).any(1).mean()
pocock = (np.abs(z) > 2.413).any(1).mean()
ob = 2.04 * np.sqrt(5 / np.arange(1, 6))
obf = (np.abs(z) > ob).any(1).mean()
print("naive 1.96 at every look:", round(naive, 4))
print("Pocock 2.413 at every look:", round(pocock, 4))
print("O'Brien-Fleming style bounds", ob.round(3), "false positive rate", round(obf, 4))
print("independent-looks formula 1-0.95^5 =", round(1 - 0.95**5, 4))

naive 1.96 at every look: 0.1413
Pocock 2.413 at every look: 0.0505
O'Brien-Fleming style bounds [4.562 3.226 2.634 2.281 2.04 ] false positive rate 0.0493
independent-looks formula 1-0.95^5 = 0.2262


In [ ]:
import numpy as np
rng = np.random.default_rng(2)
paths, n, tau2, alpha = 4000, 3000, 0.01, 0.05
x = rng.normal(size=(paths, n))                  # paired differences under the null, variance 1
k = np.arange(1, n + 1); xbar = np.cumsum(x, 1) / k
lam = np.sqrt(1 / (1 + k * tau2)) * np.exp(k**2 * tau2 * xbar**2 / (2 * (1 + k * tau2)))
print("share of null paths that EVER cross 1/alpha:", (lam.max(1) >= 1 / alpha).mean().round(4))
d = x + 0.1                                      # true effect 0.1 sd
xbar = np.cumsum(d, 1) / k
lam = np.sqrt(1 / (1 + k * tau2)) * np.exp(k**2 * tau2 * xbar**2 / (2 * (1 + k * tau2)))
hit = (lam >= 1 / alpha).argmax(1)[(lam >= 1 / alpha).any(1)] + 1
print("with a real effect of 0.1 sd: stop rate", (lam >= 1/alpha).any(1).mean().round(3), "median stop at n =", int(np.median(hit)))

share of null paths that EVER cross 1/alpha: 0.0355
with a real effect of 0.1 sd: stop rate 0.996 median stop at n = 727


## Chapter recap

## Practice set

**1. (easy)** Control rate 20%, you want to detect a 2-point lift. Use the rule of thumb for n per arm. **Answer:** 16 × 0.2 × 0.8 / 0.02² = 2.56 / 0.0004 = 6,400 per arm.

## Prompt pack

---
Done? Do the **Practice set** at the end of the chapter, then clear your Review cards on the site.

https://dadipavan.github.io/Learning/study/ds/ch07.html